# Lab 05: Fusing image tokens into a language model (LLaVA style)

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/hokwangchoi/hc-model-study/blob/master/labs/notebooks/lab05_vlm_fusion.ipynb) · lab page: [`labs/lab05.html`](../lab05.html) · file: [`labs/lab05_vlm_fusion.py`](../lab05_vlm_fusion.py)

**Time:** 45-60 min

## Goal

A vision-language model (VLM) is a ViT + a small projector + an LLM. The
ViT turns the image into patch features; the projector maps each feature
into the LLM's embedding space; the LLM then reads image tokens and text
tokens as one sequence. You build the fusion: token counting, the 2 x 2
"pixel shuffle" merge, the MLP projector, the joint sequence, and its
attention mask.

## The maths, step by step

```text
B batch, P patch size, d_v vision width, d_t LLM width (d in the decoder
code), V vocab, N_v image tokens, N_t text tokens (as in the VLM guide).

  1. How many image tokens?  N_v = (H / P) * (W / P) / merge^2
       336 x 336, P = 14         -> 24 * 24 = 576  (LLaVA-1.5)
       448 x 448, P = 14         -> 1024; with a 2 x 2 merge -> 256
     Every image token costs prefill compute and KV-cache memory, like text.
  2. 2 x 2 merge (pixel shuffle, Qwen-VL / InternVL): on the patch grid
     gh x gw, stack each 2 x 2 neighbourhood along channels:
       [B, gh*gw, d_v] -> [B, (gh/2)*(gw/2), 4*d_v]      4x fewer tokens
  3. Projector (LLaVA-1.5): z = W2 GELU(W1 v + b1) + b2
       [B, N_v, d_v] -> [B, N_v, d_t]
  4. Joint sequence: image tokens first, then the text embeddings:
       x = cat([z, E[text_ids]], dim=1)                 [B, N_v + N_t, d_t]
  5. Attention mask over the joint sequence, N = N_v + N_t
     (True = may attend):
       causal (LLaVA):     allowed[i, j] = j <= i
       image-bidirectional option (a partial prefix-LM): additionally,
                           image token i sees ALL image tokens (bidirectional
                           inside the image block). A full prefix-LM
                           (PaliGemma, pi0) makes the whole prefix (image
                           AND prompt) bidirectional; here only the image.
     Text always sees every image token (they come first), and the image
     tokens never see the text.
  6. LLM decoder on x with that mask -> logits [B, N, V]
     (the loss/next-token prediction uses only the text positions).
```

## Read alongside

- [VLM guide](../../vlm/vlm_guide.html): "The Key Idea: VLM = ViT + Projector + LLM", 1.1 Shape flow, 2. The Fusion Step (2.1 causal attention across modalities, #mask-widget), 4. Image Token Reduction
- [ViT guide](../../vit/vit_guide.html): the vision encoder (lab 04)

## How to work

1. Run the setup cell and the imports cell.
2. Implement the functions in order, replacing `raise NotImplementedError  # TODO`. The hints above each cell open one at a time; try Hint 1 first.
3. Run each cell after editing it, then run the check cell at the end: `check.run_lab(5, namespace=globals())` checks the functions defined in this notebook (TODO = not started, PASS = matches the reference, FAIL = read the message).

Your work stays in this notebook; it does not change `labs/lab05_vlm_fusion.py` or the lab page. The reference solution is `labs/solutions/lab05_vlm_fusion.py`.

In [ ]:
# Setup: make the checker (labs/check.py) importable. Run this cell first.
# In a clone of the repository (Jupyter, VS Code) it uses labs/check.py;
# elsewhere (e.g. Google Colab) it downloads check.py, which needs only torch.
import pathlib
import sys
import urllib.request


def _find_labs_dir():
    here = pathlib.Path.cwd().resolve()
    for d in [here, *here.parents]:
        for cand in (d, d / "labs"):
            if (cand / "check.py").is_file() and (cand / "solutions").is_dir():
                return cand
    return None


LABS_DIR = _find_labs_dir()
if LABS_DIR is None:
    LABS_DIR = pathlib.Path("hc_model_study_labs").resolve()
    LABS_DIR.mkdir(exist_ok=True)
    if not (LABS_DIR / "check.py").is_file():
        urllib.request.urlretrieve("https://raw.githubusercontent.com/hokwangchoi/hc-model-study/master/labs/check.py", LABS_DIR / "check.py")
sys.path.insert(0, str(LABS_DIR))
import check  # noqa: E402

import torch  # noqa: E402
print("checker:", LABS_DIR / "check.py", "| torch", torch.__version__)

### Imports

In [ ]:
from dataclasses import dataclass

import torch
import torch.nn as nn
import torch.nn.functional as F

## Step 1: count image tokens

**Hints for `count_image_tokens`** (open one at a time):

<details><summary>Hint 1</summary>

```text
patches per row = width // patch_size, per column = height // patch_size.
```

</details>

<details><summary>Hint 2</summary>

```text
check divisibility by patch_size * merge first, then divide the
patch count by merge * merge.
```

</details>

In [ ]:
def count_image_tokens(height: int, width: int, patch_size: int, merge: int = 1) -> int:
    """Number of image tokens the LLM sees for one height x width image.

    The image is cut into patch_size x patch_size patches; then every
    merge x merge block of patches becomes one token. Raise ValueError if
    height or width is not a multiple of patch_size * merge.
    """
    raise NotImplementedError  # TODO

## Step 2: the 2 x 2 merge ("pixel shuffle")

**Hints for `pixel_shuffle_merge`** (open one at a time):

<details><summary>Hint 1</summary>

```text
same idea as patchify in lab 04, but on a grid of tokens instead
of pixels: reshape to [B, grid_h // r, r, grid_w // r, r, d].
```

</details>

<details><summary>Hint 2</summary>

```text
you want [B, gh/r, gw/r, r(dy), r(dx), d] before the final reshape.
```

</details>

<details><summary>Hint 3</summary>

```text
x.reshape(B, gh // r, r, gw // r, r, d).permute(0, 1, 3, 2, 4, 5)
    .reshape(B, (gh // r) * (gw // r), r * r * d)
```

</details>

In [ ]:
def pixel_shuffle_merge(x: torch.Tensor, grid_h: int, grid_w: int, r: int = 2) -> torch.Tensor:
    """Merge every r x r block of patch tokens into one token.

    x: [B, grid_h * grid_w, d], tokens in row-major grid order.
    Returns [B, (grid_h // r) * (grid_w // r), r * r * d].

    Merged token (i, j) (row-major over the coarse grid) is the
    concatenation, in row-major (dy, dx) order, of the r*r fine tokens at
    grid positions (r*i + dy, r*j + dx).
    """
    raise NotImplementedError  # TODO

## Step 3: the projector

**Hints for `MLPProjector.forward`** (open one at a time):

<details><summary>Hint 1</summary>

```text
one line; use the exact GELU (F.gelu default).
```

</details>

In [ ]:
class MLPProjector(nn.Module):
    """LLaVA-1.5 projector: Linear(d_in, d_out) -> GELU -> Linear(d_out, d_out)."""

    def __init__(self, d_in: int, d_out: int):
        super().__init__()
        self.fc1 = nn.Linear(d_in, d_out)
        self.fc2 = nn.Linear(d_out, d_out)

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        """[B, N, d_in] -> [B, N, d_out], applied to every image token independently."""
        raise NotImplementedError  # TODO

## Step 4: the joint sequence and its attention mask

**Hints for `build_sequence`** (open one at a time):

<details><summary>Hint 1</summary>

```text
torch.cat along the sequence dim.
```

</details>

<details><summary>Hint 2</summary>

```text
is_image = torch.arange(n_img + n_txt, device=...) < n_img
```

</details>

In [ ]:
def build_sequence(image_embeds: torch.Tensor, text_embeds: torch.Tensor):
    """Image tokens first, then text.

    Args:
        image_embeds: [B, N_v, d_t]  (already projected to the LLM width)
        text_embeds:  [B, N_t, d_t]
    Returns:
        embeds:   [B, N_v + N_t, d_t]
        is_image: [N_v + N_t] bool, True at the image positions
    """
    raise NotImplementedError  # TODO

**Hints for `joint_attention_mask`** (open one at a time):

<details><summary>Hint 1</summary>

```text
start from the causal mask (lab 01): key j <= query i.
```

</details>

<details><summary>Hint 2</summary>

```text
for the image-bidirectional variant, also allow the [:n_img, :n_img] block.
```

</details>

<details><summary>Hint 3</summary>

```text
allowed = torch.ones(N, N, dtype=torch.bool, device=device).tril()
if image_bidirectional: allowed[:n_img, :n_img] = True
```

</details>

In [ ]:
def joint_attention_mask(n_img: int, n_txt: int, image_bidirectional: bool = False,
                         device=None) -> torch.Tensor:
    """[N, N] bool mask over the joint sequence, N = n_img + n_txt; True = may attend.

    image_bidirectional=False: plain causal mask (LLaVA).
    image_bidirectional=True:  causal, except that image tokens also see every
                               other image token (image-bidirectional: a partial
                               prefix-LM that covers the image only).
    """
    raise NotImplementedError  # TODO

## Given: a tiny LLM decoder that accepts an explicit attention mask

Given: `class DecoderBlock(nn.Module)` is already written. Run the cell.

In [ ]:
class DecoderBlock(nn.Module):
    def __init__(self, d: int, n_heads: int):
        super().__init__()
        self.n_heads = n_heads
        self.ln1 = nn.LayerNorm(d)
        self.qkv = nn.Linear(d, 3 * d)
        self.proj = nn.Linear(d, d)
        self.ln2 = nn.LayerNorm(d)
        self.mlp = nn.Sequential(nn.Linear(d, 4 * d), nn.GELU(), nn.Linear(4 * d, d))

    def forward(self, x: torch.Tensor, mask: torch.Tensor) -> torch.Tensor:
        B, N, d = x.shape
        q, k, v = (t.view(B, N, self.n_heads, -1).transpose(1, 2)
                   for t in self.qkv(self.ln1(x)).split(d, dim=-1))
        a = F.scaled_dot_product_attention(q, k, v, attn_mask=mask)   # mask: True = may attend
        x = x + self.proj(a.transpose(1, 2).reshape(B, N, d))
        return x + self.mlp(self.ln2(x))

Given: `class TinyDecoder(nn.Module)` is already written. Run the cell.

In [ ]:
class TinyDecoder(nn.Module):
    def __init__(self, d: int, n_heads: int, n_layers: int):
        super().__init__()
        self.blocks = nn.ModuleList(DecoderBlock(d, n_heads) for _ in range(n_layers))
        self.ln_f = nn.LayerNorm(d)

    def forward(self, x: torch.Tensor, mask: torch.Tensor) -> torch.Tensor:
        """x [B, N, d], mask [N, N] bool -> hidden states [B, N, d]"""
        for block in self.blocks:
            x = block(x, mask)
        return self.ln_f(x)

Given: `class VLMConfig` is already written. Run the cell.

In [ ]:
@dataclass
class VLMConfig:
    d_vision: int = 24
    d_model: int = 32
    vocab_size: int = 50
    n_heads: int = 4
    n_layers: int = 2

## Step 5: the fused forward pass

**Hints for `TinyVLM.forward`** (open one at a time):

<details><summary>Hint 1</summary>

```text
project the image, embed the text, build_sequence, joint_attention_mask,
self.decoder(x, mask), self.lm_head.
```

</details>

<details><summary>Hint 2</summary>

```text
create the mask on vision_feats.device.
```

</details>

In [ ]:
class TinyVLM(nn.Module):
    def __init__(self, cfg: VLMConfig):
        super().__init__()
        self.projector = MLPProjector(cfg.d_vision, cfg.d_model)
        self.tok_emb = nn.Embedding(cfg.vocab_size, cfg.d_model)
        self.decoder = TinyDecoder(cfg.d_model, cfg.n_heads, cfg.n_layers)
        self.lm_head = nn.Linear(cfg.d_model, cfg.vocab_size, bias=False)

    def forward(self, vision_feats: torch.Tensor, text_ids: torch.Tensor,
                image_bidirectional: bool = False) -> torch.Tensor:
        """
        Args:
            vision_feats: [B, N_v, d_vision]  patch features from a ViT (lab 04)
            text_ids:     [B, N_t]            token ids
        Returns:
            logits:       [B, N_v + N_t, vocab_size]
        """
        raise NotImplementedError  # TODO

## Check

Run this after each function you finish. Steps you have not started show TODO; finished steps show PASS. For a full traceback of a failing step: `check.run_lab(5, globals(), verbose=True)`.

In [ ]:
check.run_lab(5, namespace=globals());

## Scratch

Try your functions here: print shapes, call them on small inputs. (Functions you have not written yet raise `NotImplementedError`.)

In [ ]:
# Try the pieces here (Run the cells above first).
print(count_image_tokens(336, 336, 14))           # LLaVA-1.5: 576
print(count_image_tokens(448, 448, 14, merge=2))  # 256
print(joint_attention_mask(2, 3, image_bidirectional=True).int())
x = torch.randn(1, 16, 8)
pixel_shuffle_merge(x, 4, 4).shape